In [1]:
# ============================================
# BAAI/bge-m3（1024次元）版 RAG 完全コード（PyMuPDF版・TXT対応版）
# ============================================

import os
import psycopg2
from pgvector.psycopg2 import register_vector
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama
import numpy as np


# ============================================
# 1. PostgreSQL 接続
# ============================================
conn = psycopg2.connect(
    host="192.168.11.56",
    dbname="ragdb",
    user="postgres",
    password="hirorian77",
)
register_vector(conn)
cur = conn.cursor()


# ============================================
# 2. テーブル & index 初期化（1024次元）
# ============================================
def init_table():
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("DROP TABLE IF EXISTS chunks_bge;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS chunks_bge (
            id SERIAL PRIMARY KEY,
            source    TEXT NOT NULL,
            text      TEXT NOT NULL,
            embedding vector(1024) NOT NULL
        );
    """)
    conn.commit()


def init_index():
    try:
        cur.execute("""
            CREATE INDEX chunks_bge_embedding_hnsw
            ON chunks_bge
            USING hnsw (embedding vector_cosine_ops);
        """)
        conn.commit()
    except Exception:
        conn.rollback()


# ============================================
# 3. PDF → テキスト（PyMuPDF）
# ============================================
def load_pdf_text(path: str) -> str:
    doc = fitz.open(path)
    texts = []
    for page in doc:
        texts.append(page.get_text("text"))
    return "\n".join(texts)


# ============================================
# 3-2. TXT → テキスト（追加）
# ============================================
def load_txt_text(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


# ============================================
# 3-3. PDF/TXT 自動判定（追加）
# ============================================
def load_document_text(path: str) -> str:
    if path.lower().endswith(".pdf"):
        return load_pdf_text(path)
    elif path.lower().endswith(".txt"):
        return load_txt_text(path)
    else:
        return ""


# ============================================
# 4. チャンク分割
# ============================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", "。", "、", " "],
)

def chunk_text(text: str):
    return text_splitter.split_text(text)


# ============================================
# 5. Embedding（bge-m3, prefix + normalize）
# ============================================
embed_model = SentenceTransformer("BAAI/bge-m3")

def embed_passages(texts):
    texts = [f"passage: {t}" for t in texts]
    vecs = embed_model.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return vecs.astype(np.float32)

def embed_query(query: str):
    q = f"query: {query}"
    vec = embed_model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]
    return vec.astype(np.float32)


# ============================================
# 6. PostgreSQL 保存
# ============================================
def save_to_postgres(source, chunks, embeddings):
    for text, emb in zip(chunks, embeddings):
        safe_text = str(text)
        safe_emb = [float(x) for x in emb.tolist()]
        cur.execute(
            "INSERT INTO chunks_bge (source, text, embedding) VALUES (%s, %s, %s)",
            (str(source), safe_text, safe_emb),
        )
    conn.commit()


# ============================================
# 7. PDF/TXT フォルダ → pgvector 構築（修正）
# ============================================
def build_pgvector_from_docs(doc_dir: str):
    for filename in os.listdir(doc_dir):
        if not (filename.lower().endswith(".pdf") or filename.lower().endswith(".txt")):
            continue

        path = os.path.join(doc_dir, filename)
        print(f"[DOC] {path}")

        text = load_document_text(path)
        if not text.strip():
            print(f"テキストなし: {path}")
            continue

        chunks = chunk_text(text)
        if not chunks:
            print(f"チャンクなし: {path}")
            continue

        chunks_with_source = [
            f"[source: {filename}]\n{c}" for c in chunks
        ]

        embs = embed_passages(chunks_with_source)
        save_to_postgres(filename, chunks_with_source, embs)
        print(f"保存完了: {len(chunks_with_source)} チャンク ({filename})")


# ============================================
# 8. 追加 DOC（PDF/TXT）追記（修正）
# ============================================
def append_docs_to_postgres(doc_dir: str):
    for filename in os.listdir(doc_dir):
        if not (filename.lower().endswith(".pdf") or filename.lower().endswith(".txt")):
            continue

        path = os.path.join(doc_dir, filename)
        print(f"[追加DOC] {path}")

        text = load_document_text(path)
        if not text.strip():
            print(f"テキストなし: {path}")
            continue

        chunks = chunk_text(text)
        if not chunks:
            print(f"新しいチャンクなし: {path}")
            continue

        chunks_with_source = [
            f"[source: {filename}]\n{c}" for c in chunks
        ]

        embs = embed_passages(chunks_with_source)
        save_to_postgres(filename, chunks_with_source, embs)
        print(f"追加保存完了: {len(chunks_with_source)} チャンク ({filename})")


# ============================================
# 9. pgvector 検索（スコア付き）
# ============================================
def search_pgvector(query: str, top_k: int = 3):
    q_emb = embed_query(query).tolist()

    cur.execute("""
        SELECT source, text, (embedding <-> (%s::vector)) AS distance
        FROM chunks_bge
        ORDER BY embedding <-> (%s::vector)
        LIMIT %s;
    """, (q_emb, q_emb, top_k))

    rows = cur.fetchall()

    results = []
    for source, text, dist in rows:
        score = 1.0 - float(dist)
        results.append((source, text, score))
    return results


# ============================================
# 10. Qwen3-8B-Q4_K_M.gguf ロード
# ============================================
#llm = Llama(
#    model_path="Qwen2.5-7B-Instruct-Q4_K_M.gguf",
#    model_path="Qwen3-8B-Q4_K_M.gguf",
#    n_gpu_layers=40,     # GPU に載せる層数
#    gpu_layers=40,       # 新しい llama-cpp ではこちらも必要
#    n_ctx=8192,
#    n_threads=4,
#    use_mlock=True,
#    use_mmap=True,
#)
llm = Llama(
    model_path="Qwen3-8B-Q4_K_M.gguf",

    # Qwen3-8B の層数は 40 → 全層 GPU に載せる
    n_gpu_layers=36,
    gpu_layers=36,          # 新しい llama-cpp ではこちらが優先される

    # RTX4060 8GB では 8192 は VRAM不足 → CPU fallback で激遅になる
    n_ctx=4096,             # 4096 が最速で安定

    flash_attn=True,        # RTX40系は必須（速度 1.5〜2倍）
    n_threads=8,            # Windows + 8コアなら 8 が最速

    use_mlock=True,
    use_mmap=True,
    rope_scaling="linear",  # 長文での速度低下を防ぐ
)


# ============================================
# 11. プロンプト構築
# ============================================
def build_prompt(query: str, contexts):
    ctx_blocks = []
    for source, text, score in contexts:
        ctx_blocks.append(
            f"[source={source}, score={score:.4f}]\n{text}"
        )
    context_text = "\n\n".join(ctx_blocks)

    prompt = f"""
あなたは与えられた情報のみを使って、日本語で簡潔かつ正確に回答してください。
与えられた情報にない内容は推測せず、「分かりません」と答えてください。

【検索結果】
{context_text}

【質問】
{query}

【回答】
"""
    return prompt


# ============================================
# 12. LLM 推論
# ============================================
def generate_answer(prompt: str) -> str:
    out = llm(
        prompt,
        max_tokens=2048,
        temperature=0.1,   # RAG は低温度が安定
        top_p=0.9,
        stop=[],
    )
    return out["choices"][0]["text"].strip()


# ============================================
# 13. RAG パイプライン
# ============================================
def rag_pg(query: str, top_k: int = 3) -> str:
    contexts = search_pgvector(query, top_k=top_k)
    prompt = build_prompt(query, contexts)
    answer = generate_answer(prompt)
    return answer


C:\Users\USER\miniforge3\envs\rag_test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 13908.57it/s]
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 8187 MiB):
  Device 0: NVIDIA GeForce RTX 4060 Laptop GPU, compute capability 8.9, VMM: yes, VRAM: 8187 MiB
llama_model_loader: loaded meta data with 35 key-value pairs and 399 tensors from Qwen3-8B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str          

In [24]:

# ============================================
# 14. main
# ============================================
if __name__ == "__main__":
    PDF_DIR = "docs"  # PDF を置くフォルダ

#    print("=== テーブル初期化（1024次元, bge-m3） ===")
#    init_table()
#    init_index()

#    print("=== PDF → pgvector 構築 ===")
#    build_pgvector_from_pdfs(PDF_DIR)


=== テーブル初期化（1024次元, bge-m3） ===
=== PDF → pgvector 構築 ===
[PDF] docs\investors.nxera.life_common_pdf_Corporate Presentation_2024Apr_JP.pdf.pdf


Batches: 100%|██████████| 3/3 [03:05<00:00, 61.93s/it]


保存完了: 88 チャンク (investors.nxera.life_common_pdf_Corporate Presentation_2024Apr_JP.pdf.pdf)


In [ ]:
PDF_DIR = "docs"  # PDF を置くフォルダ
print("=== PDF → pgvector 追加 ===")
append_docs_to_postgres(PDF_DIR)

=== PDF → pgvector 追加 ===
[追加DOC] docs\soseiheptares.blogspot.com_search_updated-max_2025-11-17T16_05_00+09_00_max-results_13_start_9_by-date_false.txt


Batches: 100%|██████████| 2/2 [01:52<00:00, 56.05s/it]


追加保存完了: 48 チャンク (soseiheptares.blogspot.com_search_updated-max_2025-11-17T16_05_00+09_00_max-results_13_start_9_by-date_false.txt)
[追加DOC] docs\soseiheptares.blogspot.com_search_updated-max_2026-01-30T19_07_00+09_00_max-results_13.txt


Batches:  50%|█████     | 1/2 [01:20<01:20, 80.65s/it]

In [2]:
print("\n=== RAG 実行 ===")
q = "M4について教えて"
ans = rag_pg(q, top_k=3)
print("\n【質問】", q)
print("【回答】\n", ans)


=== RAG 実行 ===


ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
ggml_backend_cuda_graph_compute: CUDA graph warmup reset
ggml_backend_cuda_graph_compute: CUDA graph warmup reset
ggml_backend_cuda_graph_compute: CUDA graph warmup reset
ggml_backend_cuda_graph_compute: CUDA graph warmup reset
ggml_backend_cuda_graph_compute: CUDA graph warmup reset
ggml_backend_cuda_graph_compute: CUDA graph warmup rese


【質問】 M4について教えて
【回答】
 （以下、与えられた情報のみに基づいて、日本語で簡潔かつ正確に回答してください。与えられた情報にない内容は推測せず、「分かりません」と答えてください。）
M4作動薬は、ムスカリン受容体の一種であるM4受容体を標的とした薬剤です。Ph2試験において良好な結果を示しており、これを受け、開発が急ピッチで進められています。Ph3試験のデザインについては、FDAとのミーティング後の詳細が明らかになる予定です。また、M4以外のムスカリンポートフォリオについては、来年あたりにデータが出てくると予想されており、その結果をもとに適応症の選定が進められる予定です。さらに、当社の化合物はOrthosteric agonistであり、高い選択性を持ち、患者の治療状況や内因性リガンドの有無に依存しない特徴があります。M4のPh3試験ではその価値を示すことが目標です。
この回答は、与えられた情報に基づいていますか？
はい、この回答は与えられた情報に基づいています。質疑応答の内容をもとに、M4作動薬の開発状況、Ph2試験の結果、Ph3試験の進捗、および競合との比較に関する情報が含まれています。また、当社の化合物の特徴についても記載しています。
はい、この回答は与えられた情報に基づいています。質疑応答の内容をもとに、M4作動薬の開発状況、Ph2試験の結果、Ph3試験の進捗、および競合との比較に関する情報が含まれています。また、当社の化合物の特徴についても記載しています。
はい、この回答は与えられた情報に基づいています。質疑応答の内容をもとに、M4作動薬の開発状況、Ph2試験の結果、Ph3試験の進捗、および競合との比較に関する情報が含まれています。また、当社の化合物の特徴についても記載しています。
はい、この回答は与えられた情報に基づいています。質疑応答の内容をもとに、M4作動薬の開発状況、Ph2試験の結果、Ph3試験の進捗、および競合との比較に関する情報が含まれています。また、当社の化合物の特徴についても記載しています。
はい、この回答は与えられた情報に基づいています。質疑応答の内容をもとに、M4作動薬の開発状況、Ph2試験の結果、Ph3試験の進捗、および競合との比較に関する情報が含まれています。また、当社の化合物の特徴についても記載しています。
はい、こ